In [0]:

# NOTEBOOK: 06_business_insights

# IMPORTAR LIBRERÍAS

from pyspark.sql import functions as F
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D


# CARGAR DATOS

# IMPORTANTE:
# Antes de ejecutar este notebook,
# debe existir la tabla:
# online_retail

df_raw = spark.table("online_retail")

print("✅ Datos cargados correctamente")

df_raw.display()


# CREAR SILVER

df_silver = df_raw \
    .filter(F.col("CustomerID").isNotNull()) \
    .filter(~F.col("InvoiceNo").startswith("C")) \
    .filter(F.col("Quantity") > 0) \
    .filter(F.col("UnitPrice") > 0)


# RENOMBRAR COLUMNAS

df_silver = df_silver \
    .withColumnRenamed("InvoiceNo",   "numero_factura") \
    .withColumnRenamed("StockCode",   "codigo_producto") \
    .withColumnRenamed("Description", "descripcion") \
    .withColumnRenamed("Quantity",    "cantidad") \
    .withColumnRenamed("InvoiceDate", "fecha_factura") \
    .withColumnRenamed("UnitPrice",   "precio_unitario") \
    .withColumnRenamed("CustomerID",  "id_cliente") \
    .withColumnRenamed("Country",     "pais")


# CREAR TOTAL DE COMPRA

df_silver = df_silver.withColumn(
    "TotalAmount",
    F.col("cantidad") * F.col("precio_unitario")
)

print("✅ Silver creado correctamente")

df_silver.show(5)


# CREAR GOLD (RFM)

max_date = df_silver.select(
    F.max("fecha_factura")
).collect()[0][0]

df_rfm = df_silver.groupBy("id_cliente").agg(

    # RECENCY
    F.datediff(
        F.lit(max_date),
        F.max("fecha_factura")
    ).alias("Recency"),

    # FREQUENCY
    F.countDistinct("numero_factura").alias("Frequency"),

    # MONETARY
    F.round(
        F.sum("TotalAmount"), 2
    ).alias("Monetary")
)

print("✅ Tabla RFM creada")

df_rfm.show(10)


# ESTADÍSTICAS

print("📊 Estadísticas descriptivas")

df_rfm.describe().show()


# CONVERTIR A PANDAS

pdf = df_rfm.toPandas()

print("✅ Conversión a Pandas completada")


# VISUALIZACIÓN 2D

plt.figure(figsize=(10,6))

plt.scatter(
    pdf["Frequency"],
    pdf["Monetary"]
)

plt.xlabel("Frequency")
plt.ylabel("Monetary")
plt.title("Frecuencia de Compra vs Gasto")

plt.show()


# VISUALIZACIÓN 3D

fig = plt.figure(figsize=(10,7))

ax = fig.add_subplot(111, projection='3d')

ax.scatter(
    pdf["Recency"],
    pdf["Frequency"],
    pdf["Monetary"]
)

ax.set_xlabel("Recency")
ax.set_ylabel("Frequency")
ax.set_zlabel("Monetary")

plt.title("Visualización RFM 3D")

plt.show()


# CREAR SEGMENTOS

pdf["segmento"] = "Ocasional"

# VIP
pdf.loc[
    (pdf["Monetary"] > 5000),
    "segmento"
] = "VIP"

# En riesgo
pdf.loc[
    (pdf["Recency"] > 200),
    "segmento"
] = "En Riesgo"

# Nuevos
pdf.loc[
    (pdf["Frequency"] < 5),
    "segmento"
] = "Nuevo"

print("✅ Segmentos creados correctamente")


# DISTRIBUCIÓN DE SEGMENTOS

print("📌 Distribución de segmentos")

print(
    pdf["segmento"].value_counts()
)


# GRÁFICA DE SEGMENTOS

plt.figure(figsize=(8,5))

pdf["segmento"].value_counts().plot(kind="bar")

plt.title("Distribución de Segmentos")

plt.xlabel("Segmento")
plt.ylabel("Cantidad de Clientes")

plt.show()


# INSIGHTS DE NEGOCIO

print("""
📊 INSIGHTS DE NEGOCIO

1. Los clientes VIP generan el mayor ingreso y presentan alta frecuencia de compra.

2. Los clientes en riesgo llevan mucho tiempo sin realizar compras y requieren campañas de recuperación.

3. Los clientes nuevos tienen pocas compras registradas y necesitan incentivos de fidelización.

4. Los clientes ocasionales presentan compras esporádicas y bajo gasto promedio.
""")


# RECOMENDACIONES

print("""
💡 RECOMENDACIONES

✔️ Implementar programas de fidelización para clientes VIP.

✔️ Crear campañas de reactivación para clientes en riesgo.

✔️ Ofrecer descuentos de bienvenida para clientes nuevos.

✔️ Aplicar promociones segmentadas para clientes ocasionales.
""")

print("✅ BUSINESS INSIGHTS COMPLETADO")